# organic_ratio — entry notebook


## 1. Git sync

Подтянуть актуальный `main` из `Anton-Filimoncev-azur/organic_ratio` поверх локальной копии. Требует `GIT_PAT` в `.env`.

In [2]:
import os
import subprocess
import base64
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = "/home/jovyan/KEDRO/organic_ratio"
BRANCH = "main"
ORG = "Anton-Filimoncev-azur"
REPO = "organic_ratio"
USER = "Anton-Filimoncev-azur"

os.chdir(PROJECT_ROOT)
print("PWD:", Path().resolve())

load_dotenv()
TOKEN = os.getenv("GIT_PAT")
if not TOKEN:
    raise ValueError("GIT_PAT not found in environment")

auth = base64.b64encode(f"{USER}:{TOKEN}".encode()).decode()

subprocess.run(["git", "rebase", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "merge", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "reset", "--hard"], check=True)
subprocess.run(["git", "clean", "-fd"], check=True)

subprocess.run(
    [
        "git",
        "-c",
        f"http.extraheader=Authorization: Basic {auth}",
        "fetch",
        f"https://github.com/{ORG}/{REPO}.git",
        BRANCH,
    ],
    check=True,
)
subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], check=True)

print("Repository synced successfully.")

PWD: /home/jovyan/KEDRO/organic_ratio
HEAD is now at bf31c7d  work
HEAD is now at bf31c7d  work
Repository synced successfully.


From https://github.com/Anton-Filimoncev-azur/organic_ratio
 * branch            main       -> FETCH_HEAD


## 2. Env

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## 3. Ingestion (S3 → data/raw/partition/)

In [4]:
# %run run.py

## 4. Per-source preprocessing (raw → data/features/partition/)

In [5]:
# import gc
# %run run_preprocessing.py
# gc.collect()

## 5. Cohort aggregation

`run_cohort_aggregation.py` — мерджит per-source user-grain фичи и роллапит до cohort-grain по ключам из `cohort.keys`. Добавляет `cohort_size` и `n_calendar_days`.

In [6]:
# %run run_cohort_aggregation.py

## 6. Target + features build (model-ready train / test)

`run_target_build.py` за один проход:
1. Считает target (`organic_share`, `total_installs`, `organic_installs`) на грануляции `cohort.keys − media_source`.
2. Реагрегирует все user-grain фичи на ту же грануляцию (политика SUM/MEAN из `aggregate_to_cohort`).
3. Джойнит target + features, пишет:
   - `data/features/targets/targets.parquet` (полный + колонка `split`)
   - `data/train/targets_train.parquet` (split == train, без `split`)
   - `data/test/targets_test.parquet` (split == test, без `split`)

После этого `targets_train.parquet` / `targets_test.parquet` — это уже готовые для модели таблицы.

In [7]:
# import gc
# %run run_target_build.py
# gc.collect()

## 7. Clean (filter small cohorts)

`run_clean.py` — отрезает когорты, где `total_installs < cleaning.min_total_installs` (значение в `parameters.yml`). Пишет `targets_train_clean.parquet` и `targets_test_clean.parquet`.

In [8]:
# import gc
# %run run_clean.py
# gc.collect()

## 8. Baseline (weighted Ridge on logit)

`run_baseline.py` — линейный baseline:
- target: `logit(organic_share)`
- `sklearn.Ridge(alpha=1.0)` с `sample_weight = total_installs`
- preprocessing: numeric → median impute + StandardScaler; `platform` → one-hot; `country_code` → weighted target-encoding (mean от train); `install_date` → drop
- метрики (weighted/unweighted RMSE, MAE, R²) train + test
- top-20 коэффициентов
- сохраняет `data/predictions/baseline_{train,test}.parquet` и `data/plots/baseline_{calibration,coefficients}.png`

In [9]:
%run run_baseline.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src
Loading train: data/train/targets_train_clean.parquet
Loading test:  data/test/targets_test_clean.parquet
train: (14869, 46), test: (2069, 46)

Fitting Ridge on 39 numeric features + country_te + platform OHE

--- Metrics ---
 train: RMSE_w=0.1304  MAE_w=0.0957  R²_w=0.5229  WMAPE=28.40%  | RMSE_u=0.1691  MAE_u=0.1218
  test: RMSE_w=0.1201  MAE_w=0.0910  R²_w=0.5293  WMAPE=28.36%  | RMSE_u=0.1420  MAE_u=0.1083

--- PE distribution (train, n=14,860 non-zero targets) ---
  mean    : +10.7%
  median  : -2.8%
  Q05     : -57.7%
  Q25     : -29.2%
  Q75     : +32.0%
  Q95     : +122.8%
  within ±10%:  18.9%
  within ±20%:  34.8%
  within ±50%:  73.2%

--- PE distribution (test, n=2,069 non-zero targets) ---
  mean    : +13.7%
  median  : -0.8%
  Q05     : -49.0%
  Q25     : -23.0%
  Q75     : +36.2%
  Q95     : +119.1%
  within ±10%:  21.3%
  within ±20%:  37.8%
  within ±50%:  77.1%

--- PE buckets (train, weighted by total_installs, to

## 9. PyMC hierarchical model

`run_train.py` — иерархическая байесовская модель:

```
organic_installs[i] ~ Binomial(total_installs[i], p[i])
logit(p[i]) = α + X[i]·β + u_country[c] + u_platform[plat]
u_country  ~ Normal(0, σ_country),  σ_country ~ HalfNormal(1)
u_platform ~ Normal(0, σ_platform), σ_platform ~ HalfNormal(1)
```

Binomial-правдоподобие учитывает размер когорты — `total_installs` есть в самой функции потерь, отдельных весов не нужно.

Параметры сэмплирования в `modeling.pymc` (`parameters.yml`):
- `draws: 500`, `tune: 500`, `chains: 2`, `target_accept: 0.9`
- `nuts_sampler: pymc` (можно `numpyro` для скорости, нужен `pip install numpyro jax`)

Артефакты: `data/models/pymc/{trace.nc, prep.pkl}`, `data/predictions/pymc_{train,test}.parquet`, `data/plots/pymc_{calibration,beta}.png`.

In [10]:
%run run_train.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src


WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


[jax] default_backend: gpu
[jax] devices: [CudaDevice(id=0)]
Loading train: data/train/targets_train_clean.parquet
Loading test:  data/test/targets_test_clean.parquet
train: (14869, 46), test: (2069, 46)

Prep fitted: 39 features, 97 countries, 2 platforms

Model built on 14869 obs × 39 features
Sampling: draws=500, tune=1500, chains=2, target_accept=0.95, sampler=numpyro, chain_method=vectorized


sample: 100%|██████████| 2000/2000 [16:11<00:00,  2.06it/s]
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Saved trace: data/models/pymc/trace.nc
Saved prep:  data/models/pymc/prep.pkl

--- Sampler diagnostics ---
                 mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
alpha          -0.524  0.432  -1.270    0.368      0.106    0.077      16.0      30.0   1.16
sigma_country   1.018  0.045   0.931    1.090      0.023    0.005       4.0      26.0   1.52
sigma_platform  0.616  0.407   0.119    1.400      0.130    0.067       8.0      25.0   1.24

--- Metrics ---
 train: RMSE_w=0.1223  MAE_w=0.0915  R²_w=0.5803  WMAPE=27.15%  | RMSE_u=0.1571  MAE_u=0.1147
  test: RMSE_w=0.1158  MAE_w=0.0944  R²_w=0.5627  WMAPE=29.42%  | RMSE_u=0.1315  MAE_u=0.1050

--- PE summary (test, n=2,069) ---
      mean: +22.9%
    median: +7.5%
       q05: -41.1%
       q25: -16.3%
       q75: +44.7%
       q95: +129.6%
  within_10pct:  20.2%
  within_20pct:  37.9%
  within_50pct:  75.6%

--- PE buckets (train, weighted by total_installs, total_w=2,968,565, cohorts=14,860) ---
     

## 10. MMM data prep

`run_mmm_data.py` собирает MMM-панель `platform × country × install_date`:
- `organic_installs` (target halo-модели), `total_installs`
- `spend_<top10_sources>`, `spend_other_paid`
- `dow_0..dow_6` (dayofweek)
- `geo` = `<platform>_<country>` (dim для иерархии)

Параметры в `modeling.mmm` (`parameters.yml`): `top_n_channels`, `min_country_installs`, `adstock_l_max`, `saturation`, `geo_dim`, `yearly_seasonality`.

Артефакт: `data/features/mmm/mmm_panel.parquet`.

In [11]:
%run run_mmm_data.py

Reading installs (features): data/features/partition/installs.parquet
Reading costs (raw):         data/raw/partition/costs.parquet
  top-10 channels (non-zero spend in window): ['applovin_int', 'googleadwords_int', 'facebook ads', 'unityads_int', 'mintegral_int', 'tiktokglobal_int', 'bytedanceglobal_int', 'thespotlight_int', 'tyrads_int', 'apple search ads']
  kept top-100 (platform, country) geos → 100 actual geos
  aggregating to 7-day buckets...

Panel shape: (3100, 17)
Date range:  2024-11-01 00:00:00  →  2025-05-30 00:00:00
Geos:        100  unique (platform × country)
Channels:    10 + other_paid

organic_installs summary:
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 3100.0     │
│ null_count ┆ 0.0        │
│ mean       ┆ 370.064194 │
│ std        ┆ 814.224351 │
│ min        ┆ 2.0        │
│ 25%        ┆ 65.0       │
│ 50%        ┆ 140.0      │
│ 75%       

## 11. MMM training (halo attribution)

`run_mmm_train.py` обучает halo-MMM:

```
organic_installs[g, t] = baseline[g] + Σ_source saturation(adstock(spend_source[g, t]))
                       + Σ_dow control_dow * dow_d[t]
```

`pymc-marketing.MMM` с:
- `GeometricAdstock(l_max=7)`
- `LogisticSaturation` (можно `michaelis_menten` / `hill`)
- `dims=("geo",)` — per-(platform × country) коэффициенты

Артефакты:
- `data/models/mmm/mmm.nc` — сохранённая MMM
- `data/predictions/mmm_train.parquet`
- `data/plots/mmm_{contribution,saturation,channel_share}.png`

In [12]:
%run run_mmm_train.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src
[jax] default_backend: gpu
[jax] devices: [CudaDevice(id=0)]
Loading train panel: data/features/mmm/mmm_panel_train.parquet
Train panel: (2600, 17), geos=100, dates=2024-11-01 → 2025-04-25
Target:       organic_installs
Channels (11): ['spend_facebook ads', 'spend_tiktokglobal_int', 'spend_applovin_int', 'spend_apple search ads', 'spend_mintegral_int', 'spend_googleadwords_int', 'spend_other_paid', 'spend_bytedanceglobal_int', 'spend_unityads_int', 'spend_tyrads_int', 'spend_thespotlight_int']
Controls (0): []
Per-geo MMM: 100 geos × 26 dates

Fitting MMM: draws=1000, tune=2500, chains=4, target_accept=0.95, sampler=numpyro, chain_method=vectorized


sample: 100%|██████████| 3500/3500 [06:37<00:00,  8.81it/s]
There were 13 divergences after tuning. Increase `target_accept` or reparameterize.


Output()


Saving MMM trace → data/models/mmm/mmm.nc
Saved.

Skipping predict (set mmm.compute_predict: true to enable).
Skipping plots (set mmm.save_plots: true to enable).


## 12. MMM analysis (load trace + lightweight summaries)

`run_mmm_analyze.py` — анализ без тяжёлых `pymc-marketing.plot_*` методов:
- Load `data/models/mmm/mmm.nc` через ArviZ.
- Convergence overview (% R̂ > 1.01, min ESS).
- `az.summary` → `data/predictions/mmm_summary.csv`.
- Channel coefficients (posterior mean ± 94% HDI) → `data/plots/mmm_channel_coefs.png`.

In [13]:
%run run_mmm_analyze.py

Loading: data/models/mmm/mmm.nc
Posterior vars: ['intercept_contribution', 'adstock_alpha', 'saturation_lam', 'saturation_beta', 'y_sigma', 'channel_contribution', 'total_media_contribution_original_scale']
Coords:         {'chain': <xarray.DataArray 'chain' (chain: 4)> Size: 32B
array([0, 1, 2, 3])
Coordinates:
  * chain    (chain) int64 32B 0 1 2 3, 'draw': <xarray.DataArray 'draw' (draw: 1000)> Size: 8kB
array([  0,   1,   2, ..., 997, 998, 999], shape=(1000,))
Coordinates:
  * draw     (draw) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999, 'geo': <xarray.DataArray 'geo' (geo: 100)> Size: 4kB
array(['android_ae', 'android_am', 'android_ar', 'android_at', 'android_au',
       'android_az', 'android_bd', 'android_be', 'android_bg', 'android_br',
       'android_by', 'android_ca', 'android_ch', 'android_cl', 'android_co',
       'android_cz', 'android_de', 'android_dk', 'android_dz', 'android_eg',
       'android_es', 'android_fr', 'android_gb', 'android_ge', 'android_gr',
 

## 13. MMM out-of-sample evaluation

`run_mmm_eval.py` — берёт обученную MMM и считает прогноз на `mmm_panel_test.parquet` (период `test_start_date`..`test_end_date`):
- Test метрики: RMSE, MAE, R², WMAPE (weighted by `total_installs`)
- PE summary + buckets
- Scatter plot pred vs actual
- Predictions: `data/predictions/mmm_test.parquet`

In [14]:
%run run_mmm_eval.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src
[jax] default_backend: gpu
[jax] devices: [CudaDevice(id=0)]
Loading MMM via pymc-marketing: data/models/mmm/mmm.nc
  target_column:   organic_installs
  date_column:     install_date
  channel_columns: ['spend_facebook ads', 'spend_tiktokglobal_int', 'spend_applovin_int', 'spend_apple search ads', 'spend_mintegral_int', 'spend_googleadwords_int', 'spend_other_paid', 'spend_bytedanceglobal_int', 'spend_unityads_int', 'spend_tyrads_int', 'spend_thespotlight_int']
  control_columns: None
  dims:            ('geo',)
  output_var:      y

Test panel: (500, 17), 2025-05-02 → 2025-05-30, geos=100
X_test: (500, 13), cols=['install_date', 'geo', 'spend_facebook ads', 'spend_tiktokglobal_int', 'spend_applovin_int', 'spend_apple search ads', 'spend_mintegral_int', 'spend_googleadwords_int', 'spend_other_paid', 'spend_bytedanceglobal_int', 'spend_unityads_int', 'spend_tyrads_int', 'spend_thespotlight_int']

Running mmm.sample_posterior_predict

Sampling: [y]


Output()


Returned object type: Dataset
posterior_predictive vars: ['y']
  y dims:  ('date', 'geo', 'sample')
  y shape: (5, 100, 4000)
  after collapsing ['sample']: dims=('date', 'geo'), shape=(5, 100)

Running sample_posterior_predictive on TRAIN to derive per-geo scale...
  X_train: (2600, 13)


Sampling: [y]


Output()

  pred_train_mean: dims=('date', 'geo'), shape=(26, 100)

Derived per-geo scale (actual_train / pred_scaled_train):
  android_ae           12,944.02
  android_am           12,928.17
  android_ar           12,918.88
  android_at           12,924.81
  android_au           12,939.91
  android_az           12,910.27
  android_bd           12,925.71
  android_be           12,937.47
  android_bg           12,898.49
  android_br           12,946.95
  android_by           12,924.19
  android_ca           12,976.19
  android_ch           12,948.61
  android_cl           12,990.11
  android_co           12,921.06
  android_cz           12,925.27
  android_de           12,897.46
  android_dk           12,962.87
  android_dz           12,957.93
  android_eg           13,037.91
  android_es           12,954.27
  android_fr           12,959.62
  android_gb           12,916.98
  android_ge           12,968.29
  android_gr           12,919.52
  android_hk           12,948.77
  android_hu           12,

## 14. MMM per-channel halo attribution

`run_mmm_attribution.py` — главный бизнес-результат MMM:
- Берёт `channel_contribution[date, geo, channel]` из trace (scaled space)
- Калибрует per-geo scale через сопоставление с observed training data
- Считает halo per channel в **оригинальных install-units**
- Сравнивает с total observed organic → видим **долю органики от каждого канала**

Артефакты:
- `data/predictions/mmm_attribution.parquet` — per (date, geo, channel)
- `data/predictions/mmm_attribution_summary.csv` — per channel summary
- `data/plots/mmm_halo_per_channel.png`
- `data/plots/mmm_halo_over_time.png`

In [ ]:
# %run run_mmm_attribution.py

## (опц.) Пакетный запуск экспериментов через CONFIG_OVERRIDE_PATH

In [16]:
# import os
# from pathlib import Path
# from omegaconf import OmegaConf
#
# SWEEP_PATH = Path("conf/batch_training/sweep.yml")
# TMP_DIR = Path("conf/batch_training/.tmp")
#
# sweep_cfg = OmegaConf.load(SWEEP_PATH)
# TMP_DIR.mkdir(parents=True, exist_ok=True)
#
# for exp in sweep_cfg.experiments:
#     run_name = exp.name
#     override_path = TMP_DIR / f"{run_name}.yml"
#     OmegaConf.save(config=exp.overrides, f=override_path)
#     os.environ["CONFIG_OVERRIDE_PATH"] = str(override_path)
#     try:
#         %run run_train.py
#         %run run_eval.py
#     finally:
#         os.environ.pop("CONFIG_OVERRIDE_PATH", None)